# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DarshanBhabad/FLYRANK_Notebook/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

## 1. My lane as an ML task (type)

For content editors deciding which pages to check and update first, this is a classification task: predict whether a content item is "declining" in organic impressions over the most recent 30-day window (the repo already exposes an observed target `is_declining_label`). The output is a binary risk flag (or a probability score from a classifier that can be thresholded). We choose classification because the decision is binary (take an editorial action vs. don’t) and the target is an observed outcome in the snapshot.

In [ ]:
# This cell loads the starter CSV and shows a quick head.
import pandas as pd
pd.options.display.width = 120
pd.options.display.max_columns = 60

path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)
print("shape:", df.shape)
df.head()

## 2. Target or proxy

Target: `is_declining_label` (binary). This column is 1 when `trend_direction == "down"` (impressions last 30d fell by more than 20% vs the previous 30d window). The target is an observed outcome computed from measured impressions windows, not a synthetic label invented by a business rule — but important gotcha: `trend_direction` and `trend_pct` are derived values used to compute this label and must not be included as model features (they leak the answer).

In [ ]:
# Target distribution and explicit label-source checks
print("Columns present (sample):", [c for c in df.columns if "trend" in c or "impressions" in c][:20])

print("
is_declining_label distribution:")
print(df['is_declining_label'].value_counts(dropna=False))

print("
Check trend_pct / trend_direction existence (these are label-derived and must not be used as features):")
print("trend_pct present?", 'trend_pct' in df.columns, "trend_direction present?", 'trend_direction' in df.columns)
df[['content_id','client_id','impressions_last_30d','impressions_prev_30d','trend_pct','trend_direction','is_declining_label']].head(6)

## 3. Success metric

Primary metric: ROC AUC (probability ranking quality) because the class prevalence is moderate and we want a model that ranks higher-risk pages above lower-risk ones. Secondary metric (operational): precision@K for the top K pages editors can realistically review each cycle (example: K = 100) — this measures actionable accuracy for the human-in-the-loop workflow. A plausible "good" target to aim for before further calibration: AUC ≥ 0.70 and precision@100 substantially above the base-rate (base rate ≈ proportion of declining pages — check the dataset's value). Final thresholds should be set with the editor team.

In [ ]:
from collections import OrderedDict
stats = OrderedDict()
stats['n_rows'] = len(df)
stats['declining_count'] = int(df['is_declining_label'].sum())
stats['declining_rate'] = df['is_declining_label'].mean()
stats['unique_clients'] = df['client_id'].nunique()
stats

## 4. The unit of analysis, as a real dataframe

Unit of analysis: one row = one pseudonymized content item (one page / piece of content) aggregated over a trailing 90-day window (starter CSV) with rolling 30-day comparison columns. The row contains content metadata (keyword context, content_type), 90-day totals (impressions_90d, clicks_90d, pageviews_90d), and the 30-day windows used for label logic (impressions_last_30d, impressions_prev_30d). We will use client_id only to group/honor client-holdout splits and never as a model feature.

In [ ]:
cols_show = [
    'content_id','client_id','content_type','main_intent',
    'impressions_90d','clicks_90d','pageviews_90d',
    'impressions_last_30d','impressions_prev_30d',
    'avg_position','ctr','is_declining_label'
]
for c in cols_show:
    if c not in df.columns:
        print("Missing column:", c)
df[cols_show].head(8)

In [ ]:
# Gotcha checks
print("avg_position 0 count:", (df['avg_position'] == 0).sum())
print("
Missingness of keyword context by content_type (percent missing search_volume):")
print(df.groupby('content_type')['search_volume'].apply(lambda s: s.isna().mean()).sort_values(ascending=False))

## 5. Why ML beats a fixed rule here

A fixed rule (e.g., "flag any page with trend_pct < -20%") is brittle because:
- The same percent drop means very different absolute impact depending on baseline volume (a 30% drop on a 10-impression page is noise; on a 30k-impression page it's material).
- Clients and content types differ: some clients have sparse history, others have long panels; content_type systematically affects missingness and traffic patterns.
- Multiple signals (impressions, avg_position, recent clicks, engagement_rate, ai_traffic_pct, search_volume, content age) interact nonlinearly — a model can learn these interactions and produce a calibrated risk score.
- ML allows prioritization by likelihood (probability) instead of a hard threshold, which supports ranking pages by expected return on an editor's time.

In [ ]:
# Example: how many rows meet the simple -20% rule as defined by trend_direction == 'down'?
print("Rows with trend_direction == 'down':", (df['trend_direction'] == 'down').sum())
print("Proportion:", (df['trend_direction'] == 'down').mean())

## Self-check

Self-check before submitting:
- [ ] I filled each markdown section with the framing text and rationale.
- [ ] I included code cells that load the CSV, show shape/head, show label distribution, show the unit-of-analysis columns, and surface the avg_position / missingness gotchas.
- [ ] I DO NOT plan to use `trend_pct` or `trend_direction` as features (they are used to form the target).
- [ ] I will use client_id for grouped train/test splits (client holdout).
- [ ] I ran the notebook top-to-bottom and validated the printed checks and numbers.